## Data Exploration

- Setup Directories and data files
- perform data exploration on the input files, once successful feature can be added incrementally to "pipeline.py"

In [0]:
from pathlib import Path

project_root = Path.cwd()
print("Project Location:", project_root)

In [0]:
data_dir = project_root/"data"
print("Data Location:",data_dir)
print("Data Location Existance:",data_dir.exists())

In [0]:
if data_dir.exists():
    for path in data_dir.glob("*.csv"):
        print(path.name)
        print(path.read_text())

Create DataFrames

In [0]:
schema = "plantId string, temperature double"
observed_df = (
    spark.read.format("csv").schema(schema)
            .option("header","false")
            .load(str(data_dir) + "/plants_observed_temp.csv")
)

required_df = (
    spark.read.format("csv").schema(schema)
            .option("header","false")
            .load(str(data_dir) + "/plants_required_temp.csv")
)


observed_df.show(10)
required_df.show(10)

Average temperture for each plant

In [0]:
from pyspark.sql.functions import *

average_observed_df = observed_df.groupBy("plantId").agg(avg("temperature").alias("avg_temperature"))

In [0]:
average_observed_df.show()

Find Faulty plant

In [0]:
from pyspark.sql.functions import expr,abs

faulty_plants_df = required_df.alias("r").join(average_observed_df.alias("o"),"plantId","inner").withColumn("temp_variance",expr("o.avg_temperature - r.temperature")).withColumn("fault_ind", expr("case when abs(temp_variance) >= 5 then 1 else 0 end")).filter("fault_ind == 1").show()

In [0]:
from pyspark.sql.functions import expr,abs

faulty_plants_df = required_df.alias("r").join(average_observed_df.alias("o"),"plantId","inner").withColumn("temp_variance",expr("o.avg_temperature - r.temperature")).withColumn("fault_ind", expr("case when abs(temp_variance) >= 5 then 1 else 0 end")).filter("fault_ind == 1").select("r.plantId","o.avg_temperature")

faulty_plants_df.show()




write the datadframe to a location

In [0]:
faulty_plants_df.write.mode("overwrite").format("csv").option("header","false").save("/Volumes/workspace/default/lendingclub/faulty_plants_output")

In [0]:
%sql
select * from read_files('/Volumes/workspace/default/lendingclub/faulty_plants_output',
  format => 'csv',
  header => 'false')

In [0]:
spark.read.format("csv").option("inferSchema",True).option("header","false").load("/Volumes/workspace/default/lendingclub/faulty_plants_output").show()

## Test Python implementation

Run below cell every time incremental functionality is added to test "pipeline.py"

In [0]:
import importlib
import main.job.pipeline
importlib.reload(main.job.pipeline)

- Testing "pipeline.py" after implementing each feature from the "exploration" steps above
- Once fully tested, this would make "app.py"

In [0]:
# Setup Path
from pathlib import Path as p


project_root = p.cwd()
data_dir = project_root/"data"
src_dir = project_root/"src"

# Add src to path
import sys
if str(src_dir) not in sys.path:
    sys.path.insert(0,str(src_dir))


# Incremantal test for "pipeline.py"
from main.job.pipeline import PySparkJob

job = PySparkJob()
observed_df = job.read_csv(str(data_dir)+"/plants_observed_temp.csv")

observed_df.show()


required_df = job.read_csv(str(data_dir)+"/plants_required_temp.csv")

required_df.show()


observed_average_df = job.calc_average_temperature(observed_df)
observed_average_df.show()

faulty_df = job.find_faulty_plants(observed_average_df,required_df)
faulty_df.show()

job.save_as(faulty_df,"/Volumes/workspace/default/lendingclub/faulty_plants_output")

## Test "app.py"

In [0]:
# Setup Path
from pathlib import Path

project_root = Path.cwd()
src_dir = project_root/"src"
output_dir = "/Volumes/workspace/default/lendingclub/faulty_plants_output"

# Add src to path
import sys
if str(src_dir) not in sys.path:
    sys.path.insert(0,str(src_dir))

# Test app.py
from app import main

main(str(project_root),output_dir)

In [0]:
final_result_df = spark.read.format("csv").option("header","false").schema("plantId string,avgObservedTemperature double").load(output_dir)
final_result_df.show()